In [1]:
library(ggplot2)
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ lubridate 1.9.4     ✔ tibble    3.3.0
✔ purrr     1.2.0     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
df_plot <- read.csv('df_plot.csv', row.names = 1)

# Remove the interacting_pairs you don't want
df_plot <- df_plot %>% filter(!interacting_pair %in% c('VCAM1_integrin_a4b1_complex', 'PDCD1_PDCD1LG2'))

In [3]:
reordered_interacting_pairs <- c(
    'APP_CD74',
    'CADM1_CADM1',
    'CD28_CD80',
    'CTLA4_CD80',
    'TGFB1_TGFbeta_receptor1',
    'ANGPT2_TEK',
    'ADGRE5_CD55',
    'Dehydroepiandrosterone_bySTS_ESR1',
    'Dehydroepiandrosterone_bySTS_PPARA',
    'CADM1_NECTIN3',
    'ITGAL_ICAM1',
    'CD226_NECTIN2',
    'TNFRSF10A_TNFSF10',
    'CD46_JAG1',
    #'VCAM1_integrin_a4b1_complex',
    'PDCD1_CD274',
    #'PDCD1_PDCD1LG2',
    'SIRPG_CD47',
    'FGFR1_FGF2',
    'PDGFD_PDGFR_complex',
    'TGFB1_TGFBR3',
    'CLSTN1_NRXN3',
    'CD40LG_CD40',
    'SORL1_APP',
    'TIGIT_NECTIN2',
    'SPN_ICAM1',
    'SEMA4D_PLXNB2',
    'CD44_TYROBP',
    'SEMA4D_CD72',
    'ADGRE5_THY1',
    'Glutamate_byGLS_and_SLC1A1_Glutamate_Kainate_1_4_complex',
    'ProstaglandinE2_byPTGES2_PTGER4',
    'TNFRSF14_BTLA',
    'NOTCH1_JAG1',
    'VSIR_HLA-E',
    'CD38_PECAM1',
    'TENM1_ADGRL1',
    'SELL_SELPLG',
    'APP_SORL1',
    'CD28_CD86',
    'CTLA4_CD86',
    'CD2_CD58',
    'CD6_ALCAM',
    'IGF1R_IGF1',
    'TGFB1_TGFbeta_receptor2',
    'TGFBR3_TGFB1',
    'TGFbeta_receptor1_TGFB1',
    'TGFbeta_receptor2_TGFB1',
    'ADGRE5_integrin_aVb3_complex'
)

In [4]:
tmp_aitl <- df_plot %>% filter(diagnosis == 'TFHL') %>%
    group_by(interacting_pair, tme) %>%
    mutate(num_sgnf = sum(p < 0.05)) %>%
    select(interacting_pair, tme, diagnosis, num_sgnf) %>% unique
N_aitl <- df_plot %>% filter(diagnosis == 'TFHL') %>% pull(celltype) %>% unique %>% length
tmp_aitl$perc_sgnf <- tmp_aitl$num_sgnf / N_aitl

tmp_nos <- df_plot %>% filter(diagnosis == 'NOS') %>%
    group_by(interacting_pair, tme) %>%
    mutate(num_sgnf = sum(p < 0.05)) %>%
    select(interacting_pair, tme, diagnosis, num_sgnf) %>% unique
N_nos <- df_plot %>% filter(diagnosis == 'NOS') %>% pull(celltype) %>% unique %>% length
tmp_nos$perc_sgnf <- tmp_nos$num_sgnf / N_nos

tmp_ebv <- df_plot %>% filter(diagnosis == 'EBV') %>%
    group_by(interacting_pair, tme) %>%
    mutate(num_sgnf = sum(p < 0.05)) %>%
    select(interacting_pair, tme, diagnosis, num_sgnf) %>% unique
N_ebv <- df_plot %>% filter(diagnosis == 'EBV') %>% pull(celltype) %>% unique %>% length
tmp_ebv$perc_sgnf <- tmp_ebv$num_sgnf / N_ebv

tmp_tfh <- df_plot %>% filter(diagnosis == 'TFH') %>%
    group_by(interacting_pair, tme) %>%
    mutate(num_sgnf = sum(p < 0.05)) %>%
    select(interacting_pair, tme, diagnosis, num_sgnf) %>% unique
N_tfh <- df_plot %>% filter(diagnosis == 'TFH') %>% pull(celltype) %>% unique %>% length
tmp_tfh$perc_sgnf <- tmp_tfh$num_sgnf / N_tfh

tmp_CD4 <- df_plot %>% filter(diagnosis == 'CD4') %>%
    group_by(interacting_pair, tme) %>%
    mutate(num_sgnf = sum(p < 0.05)) %>%
    select(interacting_pair, tme, diagnosis, num_sgnf) %>% unique
N_CD4 <- df_plot %>% filter(diagnosis == 'CD4') %>% pull(celltype) %>% unique %>% length
tmp_CD4$perc_sgnf <- tmp_CD4$num_sgnf / N_CD4



tmp <- rbind(tmp_aitl, tmp_nos, tmp_tfh, tmp_CD4)
tmp$tme <- factor(tmp$tme, levels = c('CD4', 'CD8', 'Treg', 'macrophage', 'monocyte', 'cDC', 'pDC',
                                'B.cell', 'GCB', 'plasma', 'stromal', 'endothelium'))

options(repr.plot.width = 18, repr.plot.height = 10)
p_heatmap <- ggplot(tmp, aes(x = diagnosis, y = interacting_pair, fill = perc_sgnf)) +
    geom_tile() +
    facet_wrap(. ~ tme, nrow = 1) +
    scale_x_discrete(limits = c('CD4', 'TFH', 'TFHL', 'NOS')) +
    scale_y_discrete(limits = reordered_interacting_pairs %>% rev) +
    scale_fill_gradient(low = "gray", high = "dark red") +
    theme(
        axis.text.x = element_text(angle = 90, hjust = 1) # Rotate 90 degrees, align right
    ) +
    xlab('') + ylab('')

#p_heatmap


#pdf('plot_heatmap_perc_sgnf.pdf', width = 18, height = 10)
#p_heatmap
#dev.off()